In [38]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import os
import pandas as pd
import numpy as np
import patsy

from functools import partial

In [27]:
jax.config.update("jax_enable_x64", True)

# 1. Data Preparation

In [67]:
# 1. Load data
df = pd.read_csv("/Users/yumin/Documents/Research/PML/Repository/amip-robustness-jax/data/113289-V1-2/processed_CT_data.csv", encoding='latin-1')

# 2. Apply outlier truncation
df = df[df['Cind'] < 10000].copy()

In [72]:
df.columns

Index(['folio', 'entidad', 'mpio', 'local', 't', 'Cind', 'hhhage', 'hhhsex',
       'p16', 'hhhalpha', 'hhhspouse', 'yycali_1', 'region_1', 'indice',
       'hectareas', 'treatp', 'treatnp', 'treatd', 'treatap', 'vhhnum',
       'avg_shock', 'village'],
      dtype='str')

# 2. OLS

In [78]:
# 3. Define OLS
@partial(jax.jit, static_argnames=['num_clusters'])
def fit_ols_clustered(X, y, weights, cluster_ids, num_clusters):
    """
    Computes OLS coefficients and Cluster-Robust Standard Errors using JAX.
    Utilizes pseudo-inverse to handle rank-deficient matrices and computes
    CRV0 standard errors (no finite-sample correction) to match the R target.
    """
    # 1. Apply weights to the design matrix
    X_w = X * weights[:, None]
    
    # 2. Calculate beta
    XTX = jnp.dot(X_w.T, X)
    XTy = jnp.dot(X_w.T, y)
    
    # Pseudo-inverse to bypass singular matrix (nan) errors
    XTX_pinv = jnp.linalg.pinv(XTX)
    beta = jnp.dot(XTX_pinv, XTy)

    # 3. Calculate true residuals
    eps = y - jnp.dot(X, beta)

    # 4. Calculate score vectors for each observation 
    # Scale each row of X by its true residual AND its AMIP weight
    X_eps = X * (weights * eps)[:, None]
    
    # 5. Sum the scores by cluster to get the "Meat"
    cluster_scores = jax.ops.segment_sum(X_eps, cluster_ids, num_segments=num_clusters)
    B = jnp.dot(cluster_scores.T, cluster_scores)
    
    # 6. The "Bread" of the sandwich is the inverted weighted XTX matrix
    bread = XTX_pinv
    
    # 7. CRV0 Variance-Covariance Matrix (NO finite-sample df correction)
    cov_matrix = jnp.dot(jnp.dot(bread, B), bread)
    
    # 8. Extract standard errors
    se = jnp.sqrt(jnp.diag(cov_matrix))

    return beta, se

# 4. Iterate over the time perios and treatments to recreate Table 1
time_periods = [8, 9, 10]
treatments = {'treatp':'Poor', 'treatnp':'Non-poor'}
base_formula = "Cind ~ {treatment} + avg_shock + hhhage + C(hhhsex) + C(p16) + C(hhhalpha) + C(hhhspouse) + yycali_1 + C(region_1) + indice + hectareas + vhhnum"
cluster_col = 'village' 

for treatment in treatments:
    for t in time_periods:
        # Copy the subset to avoid SettingWithCopy warnings
        df_subset = df[df['t'] == t].copy()

        formula = base_formula.format(treatment=treatment)

        # patsy drops rows with NaNs, returning a smaller dataframe
        y_pd, X_pd = patsy.dmatrices(formula, df_subset, return_type='dataframe')

        X_jax = jnp.array(X_pd.values)
        y_jax = jnp.array(y_pd.values).squeeze()

        weights = jnp.ones(y_jax.shape[0])

        # ALIGNMENT FIX: Extract the cluster column only for the rows patsy kept
        aligned_clusters = df_subset.loc[X_pd.index, cluster_col]

        # Encode aligned cluster IDs into contiguous integers starting at 0
        cluster_factors, _ = pd.factorize(aligned_clusters)
        cluster_ids = jnp.array(cluster_factors)
        num_clusters = len(np.unique(cluster_factors))

        # Fit the model using the clustered function
        beta, se = fit_ols_clustered(X_jax, y_jax, weights, cluster_ids, num_clusters=num_clusters)

        treatment_idx = X_pd.columns.get_loc(treatment)
        treat_beta = beta[treatment_idx]
        treat_se = se[treatment_idx]

        print(f"Time: {t} | Treatment: {treatments[treatment]}")
        print(f"  Estimate: {treat_beta:,.3f}")
        print(f"  Std. Err: {treat_se:,.3f}")
        print("-" * 40)

Time: 8 | Treatment: Poor
  Estimate: 17.312
  Std. Err: 4.576
----------------------------------------
Time: 9 | Treatment: Poor
  Estimate: 27.924
  Std. Err: 5.770
----------------------------------------
Time: 10 | Treatment: Poor
  Estimate: 33.861
  Std. Err: 4.468
----------------------------------------
Time: 8 | Treatment: Non-poor
  Estimate: -5.444
  Std. Err: 7.133
----------------------------------------
Time: 9 | Treatment: Non-poor
  Estimate: 22.852
  Std. Err: 10.000
----------------------------------------
Time: 10 | Treatment: Non-poor
  Estimate: 21.493
  Std. Err: 9.405
----------------------------------------


# 3. AMIP JAX Implementation

In [76]:
@jax.jit
def G(betas, x_n, y_n):
    """
    The gradient of the squared error for one observation.
    For an unweighted OLS model, this requires no additional weight parameter.
    """
    residual = y_n - jnp.dot(x_n, betas)
    gradient = -residual * x_n
    return gradient

In [85]:
@jax.jit
def compute_influence_scores(beta_hat, X, y, weights, param_index):
    """
    Compute the influence score for every data point.

    Args:
        beta_hat: Array of shape (P,) representing the baseline OLS estimates.
        X: Matrix of shape (N, P) containing covariates.
        y: Array of shape (N,) containing the outcome variable.
        weights: Array of shape (N,) containing AMIP perturbation weights.
        param_index: Integer index of the parameter we care about.

    Returns:
        psi: Array of shape (N,) containing the influence score for each observation.
    """
    N, P = X.shape

    # 1. Compute the Hessian (Derivative of G_cash_transfers)
    hessian_G = jax.jacfwd(G, argnums=0)

    # Vectorize the functions so they can evaluate the whole dataset at once.
    G_vmap = jax.vmap(G, in_axes=(None, 0, 0))
    hessian_G_vmap = jax.vmap(hessian_G, in_axes=(None, 0, 0))

    # Evaluate G and the Hessian for every single data point at the baseline beta_hat
    G_vals = G_vmap(beta_hat, X, y)         # Shape: (N, P)
    H_vals = hessian_G_vmap(beta_hat, X, y) # Shape: (N, P, P)

    # 2. Aggregate and Invert the Hessian
    H_sum = jnp.sum(H_vals, axis=0)         # Shape: (P, P)
    
    # Use pseudo-inverse to bypass singular matrix (nan) errors caused by dummy variables
    H_inv = jnp.linalg.pinv(H_sum)          # Shape: (P, P)

    # 3. Calculate the influence score
    # Define the derivative of our quantity of interest \phi(\beta)
    # Because \phi(\beta) = \beta_1, its derivative is a vector of zeros with a 1 at param_index
    d_phi = jnp.zeros(P).at[param_index].set(1.0)

    # Calculate the right side of the chain rule equation: H_inv @ d_phi
    # Shape will be (P,)
    chain_rule_vector = jnp.dot(H_inv.T, d_phi)

    # Multiply by G(\beta, d_n) for every person to get their specific \psi_n
    # The negative sign applies the chain rule formula from the AMIP implicit function theorem
    # Shape will be (N,)
    psi = -jnp.dot(G_vals, chain_rule_vector)

    return psi

In [86]:
def find_amip_subset(psi_scores, beta_estimate, standard_error, target_type):
    """
    Finds the indices of the data points to drop to achieve a specific target.

    target_type can be: 'sign_change', 'significance_change', 'sign_reversal'.
    """
    N = len(psi_scores)

    # 1. Define the magnitude of change (Delta) required for our specific goal
    abs_beta = jnp.abs(beta_estimate)
    margin_of_error = 1.96 * standard_error

    if target_type == 'sign_change':
        delta = abs_beta
    elif target_type == 'significance_change':
        delta = abs_beta - margin_of_error
        if delta <= 0:
            print("Estimate is already statistically insignificant!")
            return jnp.array([]), 0.0
    elif target_type == 'sign_reversal':
        delta = abs_beta + margin_of_error
    else:
        raise ValueError("Invalid target_type")

    # 2. Sort the influence scores
    # If beta > 0, we want to decrease it, so we drop points with positive influence.
    # If beta < 0, we want to increase it, so we drop points with negative influence.
    if beta_estimate > 0:
        sort_indices = jnp.argsort(-psi_scores)
        direction_multiplier = -1
    else:
        sort_indices = jnp.argsort(psi_scores)
        direction_multiplier = 1
    
    sorted_psi = psi_scores[sort_indices]

    # 3. Iteratively accumulate the change
    incremental_change = -sorted_psi * direction_multiplier
    
    # THEORY ALIGNMENT: Only accumulate drops that push the estimate in the target direction
    valid_mask = incremental_change > 0
    cumulative_magnitude_change = jnp.cumsum(incremental_change * valid_mask)

    threshold_met_mask = cumulative_magnitude_change >= delta

    if not jnp.any(threshold_met_mask):
        print(f"Target '{target_type}' cannot be reached even if we drop all helpful data.")
        return None, None
    
    # Cast M to Python int to ensure JAX slicing compatibility outside of JIT
    M = int(jnp.argmax(threshold_met_mask)) + 1
    alpha = M / N

    indices_to_drop = sort_indices[:M]

    return indices_to_drop, alpha

In [87]:
def run_cash_transfers_amip_analysis(y, X, weights, clusters, num_clusters, param_index):
    """
    Executes the complete AMIP pipeline for the Cash Transfers model using JAX.
    """
    N = y.shape[0]

    # Step 0: Baseline Estimation
    baseline_beta, baseline_se = fit_ols_clustered(X, y, weights, clusters, num_clusters)

    target_beta = baseline_beta[param_index]
    target_se = baseline_se[param_index]

    # Step 1: Calculate Influence Scores
    psi_scores = compute_influence_scores(
        beta_hat=baseline_beta.flatten(),
        X=X,
        y=y.flatten(),
        weights=weights.flatten(),
        param_index=param_index
    )

    targets = ['sign_change', 'significance_change', 'sign_reversal']

    for target in targets:
        print(f"--- Target: {target.upper()} ---")

        # Steps 2 & 3: Find the subset size (M)
        drop_indices, alpha = find_amip_subset(psi_scores, target_beta, target_se, target)

        if alpha is None or len(drop_indices) == 0:
            continue

        print(f"Points to drop (M): {len(drop_indices)}")
        print(f"Proportion (alpha): {alpha*100:.2f}%")

        # Step 4: The Validation Refit 
        # Update the specific AMIP weights to 0.0 for the targeted subset
        refit_weights = weights.at[drop_indices].set(0.0)

        # Re-run the model with the dropped points
        refit_beta, refit_se = fit_ols_clustered(X, y, refit_weights, clusters, num_clusters)

        actual_new_estimate = refit_beta[param_index]
        actual_new_se = refit_se[param_index]

        print(f"Validation Refit Estimate: {actual_new_estimate:.3f} | Refit SE: {actual_new_se:.3f}\n")

In [88]:
# 4. Iterate over time periods and treatments to execute the AMIP analysis
time_periods = [8, 9, 10]
treatments = {'treatp': 'Poor', 'treatnp': 'Non-poor'}
base_formula = "Cind ~ {treatment} + avg_shock + hhhage + C(hhhsex) + C(p16) + C(hhhalpha) + C(hhhspouse) + yycali_1 + C(region_1) + indice + hectareas + vhhnum"
cluster_col = 'village'

for treatment_var, treatment_label in treatments.items():
    for t in time_periods:
        print(f"============================================================")
        print(f"Executing AMIP for Time: {t} | Treatment: {treatment_label}")
        print(f"============================================================")
        
        # Copy the subset to avoid SettingWithCopy warnings
        df_subset = df[df['t'] == t].copy()

        # Format the regression formula for the specific treatment
        formula = base_formula.format(treatment=treatment_var)

        # patsy drops rows with NaNs, returning a cleaned dataframe
        y_pd, X_pd = patsy.dmatrices(formula, df_subset, return_type='dataframe')

        # Convert to JAX arrays
        X_jax = jnp.array(X_pd.values)
        y_jax = jnp.array(y_pd.values).squeeze()

        # Initialize AMIP weights to 1.0 for all baseline observations
        weights_jax = jnp.ones(y_jax.shape[0])

        # Extract the cluster column only for the rows patsy kept to maintain alignment
        aligned_clusters = df_subset.loc[X_pd.index, cluster_col]

        # Encode aligned cluster IDs into contiguous integers starting at 0
        cluster_factors, _ = pd.factorize(aligned_clusters)
        clusters_jax = jnp.array(cluster_factors)
        num_clusters = len(np.unique(cluster_factors))

        # Dynamically find the integer index of the treatment parameter
        param_index = X_pd.columns.get_loc(treatment_var)

        # Execute the AMIP pipeline
        run_cash_transfers_amip_analysis(
            y=y_jax,
            X=X_jax,
            weights=weights_jax,
            clusters=clusters_jax,
            num_clusters=num_clusters,
            param_index=param_index
        )
        print("\n")

Executing AMIP for Time: 8 | Treatment: Poor
--- Target: SIGN_CHANGE ---
Points to drop (M): 252
Proportion (alpha): 2.30%
Validation Refit Estimate: -0.656 | Refit SE: 3.745

--- Target: SIGNIFICANCE_CHANGE ---
Points to drop (M): 58
Proportion (alpha): 0.53%
Validation Refit Estimate: 8.755 | Refit SE: 3.983

--- Target: SIGN_REVERSAL ---
Points to drop (M): 572
Proportion (alpha): 5.23%
Validation Refit Estimate: -10.614 | Refit SE: 3.637



Executing AMIP for Time: 9 | Treatment: Poor
--- Target: SIGN_CHANGE ---
Points to drop (M): 345
Proportion (alpha): 3.58%
Validation Refit Estimate: -1.377 | Refit SE: 4.406

--- Target: SIGNIFICANCE_CHANGE ---
Points to drop (M): 83
Proportion (alpha): 0.86%
Validation Refit Estimate: 10.905 | Refit SE: 4.575

--- Target: SIGN_REVERSAL ---
Points to drop (M): 756
Proportion (alpha): 7.85%
Validation Refit Estimate: -14.675 | Refit SE: 4.439



Executing AMIP for Time: 10 | Treatment: Poor
--- Target: SIGN_CHANGE ---
Points to drop (M): 697
Pro